// 25AIML060 //

NIYATI RAIYANI //

AIUC202_DATA ANALYSIS WITH PYTHON //

CSPIT-AIML //

BATCH-C //

Project 1 :- ShoppingCart //

//

# Building a GUI-Based Shopping Cart System in Python

**Student Name:** Niyati Raiyani  
**Roll Number:** 25AIML060  
**Project Name:** Shopping Cart System

## Introduction
The aim of this project is to make a simple shopping cart application using Python and Tkinter. It lets a user view products, add selected items to a cart, search and sort products, apply a discount, export a bill, and complete checkout. The notebook is arranged step by step according to the project instructions.


## Step 1: Import Libraries

These imports provide the GUI, pop-up messages, file selection, and CSV file handling.


In [1]:
import tkinter as tk
from tkinter import messagebox, ttk, filedialog
import csv


## Step 2: Sample Product List and User Data

The product catalogue stores each product in a dictionary. The users dictionary is for this practice example, and the cart starts empty.


In [2]:
products = [
    {"id": 1, "name": "Laptop", "category": "Electronics", "price": 800.00},
    {"id": 2, "name": "T-Shirt", "category": "Clothing", "price": 20.00},
    {"id": 3, "name": "Book", "category": "Stationery", "price": 10.00},
    {"id": 4, "name": "Headphones", "category": "Electronics", "price": 50.00},
    {"id": 5, "name": "Coffee Mug", "category": "Kitchen", "price": 8.00},
]
users = {"admin": "password"}  # Practice example only; never store real passwords as plain text.
cart = []


## Step 3: Login Module

This function reads the username and password and opens the main application when the details match.


In [3]:
def login():
    username = username_entry.get().strip()
    password = password_entry.get()
    if users.get(username) == password:
        login_window.destroy()
        open_main_app()
    else:
        messagebox.showerror("Login Failed", "Invalid username or password. Please try again.")


## Step 4: Add to Cart Module

This function checks the selected products, adds them to the cart, and refreshes the display.


In [4]:
def add_to_cart():
    selected = product_box.selection()
    if not selected:
        messagebox.showwarning("No Selection", "Please select at least one product.")
        return
    for row_id in selected:
        values = product_box.item(row_id, "values")
        name, category = values[0], values[1]
        # Find the original catalogue record to retain its numeric price.
        original = next(
            (p for p in products if p["name"] == name and p["category"] == category),
            None
        )
        if original is not None:
            cart.append({
                "id": original["id"],
                "name": original["name"],
                "category": original["category"],
                "price": float(original["price"])
            })
    refresh_cart()


## Step 5: Refresh Cart Function

This function redraws the cart table and updates the total amount.


In [5]:
def refresh_cart():
    """Redraw the cart table and update the displayed total."""
    for row in cart_box.get_children():
        cart_box.delete(row)
    for item in cart:
        cart_box.insert(
            "", "end",
            values=(item["name"], item["category"], f"Rs. {item['price']:.2f}")
        )
    total_label.config(text=f"Total: Rs. {calculate_total():.2f}")


## Step 6: Calculate Total Function

This function adds the prices of all items currently in the cart.


In [6]:
def calculate_total():
    """Return the sum of all prices currently in the cart."""
    return sum(float(item["price"]) for item in cart)


## Step 7: Sort Cart Function

The cart can be sorted by price or by category.


In [7]:
def sort_cart(by):
    if by == "price":
        cart.sort(key=lambda item: item["price"])
    elif by == "category":
        cart.sort(key=lambda item: item["category"].lower())
    refresh_cart()


## Step 8: Export Bill Function

This function saves the cart details in a CSV file selected by the user.


In [8]:
def export_bill():
    if not cart:
        messagebox.showwarning("Empty Cart", "There are no items to export.")
        return
    file_path = filedialog.asksaveasfilename(
        defaultextension=".csv",
        filetypes=[("CSV Files", "*.csv")],
        title="Save Shopping Bill"
    )
    if not file_path:
        return
    try:
        with open(file_path, "w", newline="", encoding="utf-8") as file:
            writer = csv.writer(file)
            writer.writerow(["Name", "Category", "Price"])
            for item in cart:
                writer.writerow([item["name"], item["category"], f"{item['price']:.2f}"])
        messagebox.showinfo("Bill Exported", f"Bill saved successfully:\n{file_path}")
    except OSError as error:
        messagebox.showerror("Export Error", f"Could not save the bill:\n{error}")


## Step 9: Function Display Product

This function clears the previous product rows and displays the supplied product list.


In [9]:
def display_products(product_list):
    """Display the supplied products in the product table."""
    for row in product_box.get_children():
        product_box.delete(row)
    for product in product_list:
        product_box.insert(
            "", "end",
            values=(product["name"], product["category"], f"{product['price']:.2f}"),
            tags=(str(product["id"]),)
        )


## Step 10: Search Product Function

The search checks product names and categories without considering uppercase or lowercase letters.


In [10]:
def search_product():
    query = search_entry.get().strip().lower()
    results = [
        product for product in products
        if query in product["name"].lower() or query in product["category"].lower()
    ]
    display_products(results)
    if not results:
        messagebox.showinfo("Search Results", "No matching products found.")


## Step 11: Apply Discount Function

The discount is checked to ensure it is between 0 and 100 before prices are updated.


In [11]:
def apply_discount():
    try:
        rate = float(discount_entry.get().strip())
        if not 0 <= rate <= 100:
            raise ValueError
    except ValueError:
        messagebox.showerror("Invalid Discount", "Enter a discount from 0 to 100.")
        return
    for item in cart:
        item["price"] *= (1 - rate / 100)
    refresh_cart()


## Step 12: Checkout Function

Checkout displays the total bill and clears the cart after a successful purchase.


In [12]:
def checkout():
    if not cart:
        messagebox.showinfo("Checkout", "Cart is empty. Please add products first.")
        return
    total_amount = calculate_total()
    messagebox.showinfo(
        "Checkout Complete",
        f"Your total bill is Rs. {total_amount:.2f}.\nThank you for shopping!"
    )
    cart.clear()
    refresh_cart()


## Step 13: Main Window of Shopping Cart App

This function creates the product table, cart table, total label, and action controls.


In [13]:
def open_main_app():
    global product_box, cart_box, total_label, discount_entry, search_entry

    app = tk.Tk()
    app.title("Shopping Cart System")
    app.geometry("900x480")

    tk.Label(app, text="Available Products", font=("Arial", 12, "bold")).grid(
        row=0, column=0, padx=10, pady=8
    )
    product_box = ttk.Treeview(
        app, columns=("Name", "Category", "Price"), show="headings", selectmode="extended"
    )
    for col in ("Name", "Category", "Price"):
        product_box.heading(col, text=col)
        product_box.column(col, width=150)
    product_box.grid(row=1, column=0, padx=10, pady=5, sticky="nsew")
    display_products(products)

    tk.Label(app, text="Shopping Cart", font=("Arial", 12, "bold")).grid(
        row=0, column=1, padx=10, pady=8
    )
    cart_box = ttk.Treeview(app, columns=("Name", "Category", "Price"), show="headings")
    for col in ("Name", "Category", "Price"):
        cart_box.heading(col, text=col)
        cart_box.column(col, width=150)
    cart_box.grid(row=1, column=1, padx=10, pady=5, sticky="nsew")

    total_label = tk.Label(app, text="Total: Rs. 0.00", font=("Arial", 11, "bold"))
    total_label.grid(row=2, column=1, padx=10, pady=5, sticky="w")

    controls_left = tk.Frame(app)
    controls_left.grid(row=2, column=0, rowspan=4, padx=10, pady=8, sticky="nw")
    tk.Button(controls_left, text="Add to Cart", command=add_to_cart, width=18).pack(pady=3)
    tk.Button(controls_left, text="Sort by Price", command=lambda: sort_cart("price"), width=18).pack(pady=3)
    tk.Button(controls_left, text="Sort by Category", command=lambda: sort_cart("category"), width=18).pack(pady=3)
    tk.Button(controls_left, text="Export Bill", command=export_bill, width=18).pack(pady=3)
    tk.Button(controls_left, text="Checkout", command=checkout, width=18).pack(pady=3)

    search_frame = tk.Frame(app)
    search_frame.grid(row=3, column=1, padx=10, pady=5, sticky="w")
    tk.Label(search_frame, text="Search name/category:").pack(anchor="w")
    search_entry = tk.Entry(search_frame, width=25)
    search_entry.pack(side="left", padx=(0, 5))
    tk.Button(search_frame, text="Search", command=search_product).pack(side="left")
    tk.Button(search_frame, text="Show All", command=lambda: display_products(products)).pack(side="left", padx=5)

    discount_frame = tk.Frame(app)
    discount_frame.grid(row=4, column=1, padx=10, pady=5, sticky="w")
    tk.Label(discount_frame, text="Discount (%)").pack(anchor="w")
    discount_entry = tk.Entry(discount_frame, width=15)
    discount_entry.pack(side="left", padx=(0, 5))
    tk.Button(discount_frame, text="Apply Discount", command=apply_discount).pack(side="left")

    app.grid_columnconfigure(0, weight=1)
    app.grid_columnconfigure(1, weight=1)
    app.grid_rowconfigure(1, weight=1)
    app.mainloop()


## Step 14: Build Login Window

The login window is shown first. The password is hidden while typing, and the Login button calls the login function.


In [14]:
login_window = tk.Tk()

login_window.title("Shopping Cart Login")

login_window.geometry("300x190")

tk.Label(login_window, text="Username").pack(pady=(15, 2))

username_entry = tk.Entry(login_window)

username_entry.pack()

tk.Label(login_window, text="Password").pack(pady=(8, 2))

password_entry = tk.Entry(login_window, show="*")

password_entry.pack()

tk.Button(login_window, text="Login", command=login).pack(pady=12)

login_window.mainloop()


## Step 15: Execute the Code

1. Save or download this notebook as a Python (`.py`) file using Jupyter’s download/export option.
2. If needed, remove notebook-only comment lines from the exported script.
3. Open Command Prompt in the folder where the file is saved.
4. Run the program using `python 25AIML060_ShoppingCart.py`.

**Testing login:** Username: `admin`  |  Password: `password`

These are sample credentials for this practice project only.
